# Unit 8: Proximal Policy Gradient (PPO) with PyTorch 🤖

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit9/thumbnail.png" alt="Unit 8"/>


In this notebook, you'll learn to **code your PPO agent from scratch with PyTorch using CleanRL implementation as model**.

To test its robustness, we're going to train it in:

- [LunarLander-v2 🚀](https://www.gymlibrary.dev/environments/box2d/lunar_lander/)


⬇️ Here is an example of what you will achieve. ⬇️

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/sb3/ppo-LunarLander-v2/resolve/main/replay.mp4" type="video/mp4"></video>

We're constantly trying to improve our tutorials, so **if you find some issues in this notebook**, please [open an issue on the GitHub Repo](https://github.com/huggingface/deep-rl-class/issues).

## Objectives of this notebook 🏆

At the end of the notebook, you will:

- Be able to **code your PPO agent from scratch using PyTorch**.
- Be able to **push your trained agent and the code to the Hub** with a nice video replay and an evaluation score 🔥.




## This notebook is from the Deep Reinforcement Learning Course
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

In this free course, you will:

- 📖 Study Deep Reinforcement Learning in **theory and practice**.
- 🧑‍💻 Learn to **use famous Deep RL libraries** such as Stable Baselines3, RL Baselines3 Zoo, CleanRL and Sample Factory 2.0.
- 🤖 Train **agents in unique environments**

Don’t forget to **<a href="http://eepurl.com/ic5ZUD">sign up to the course</a>** (we are collecting your email to be able to **send you the links when each Unit is published and give you information about the challenges and updates).**


The best way to keep in touch is to join our discord server to exchange with the community and with us 👉🏻 https://discord.gg/ydHrjt3WP5

## Prerequisites 🏗️
Before diving into the notebook, you need to:

🔲 📚 Study [PPO by reading Unit 8](https://huggingface.co/deep-rl-course/unit8/introduction) 🤗  

To validate this hands-on for the [certification process](https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process), you need to push one model, we don't ask for a minimal result but we **advise you to try different hyperparameters settings to get better results**.

If you don't find your model, **go to the bottom of the page and click on the refresh button**

For more information about the certification process, check this section 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process

## Set the GPU 💪
- To **accelerate the agent's training, we'll use a GPU**. To do that, go to `Runtime > Change Runtime type`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">

- `Hardware Accelerator > GPU`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">

## Create a virtual display 🔽

During the notebook, we'll need to generate a replay video. To do so, with colab, **we need to have a virtual screen to be able to render the environment** (and thus record the frames).

Hence the following cell will install the librairies and create and run a virtual screen 🖥

In [ ]:
import sys

print("Instalando solo lo necesario para PPO con CartPole...")

!{sys.executable} -m pip install -q --upgrade pip setuptools wheel
!{sys.executable} -m pip install -q "gymnasium==1.0.0"
!{sys.executable} -m pip install -q tensorboard huggingface_hub wasabi imageio imageio-ffmpeg

print("LISTO. No reinicies. Sigue con la prueba.")

Instalando solo lo necesario para PPO con CartPole...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
LISTO. No reinicies. Sigue con la prueba.


In [ ]:
import gymnasium as gym
import torch
import numpy as np

print("Gymnasium:", gym.__version__)
print("Torch:", torch.__version__)
print("NumPy:", np.__version__)
print("CUDA disponible:", torch.cuda.is_available())

env = gym.make("CartPole-v1")
obs, info = env.reset()

action = env.action_space.sample()
obs, reward, terminated, truncated, info = env.step(action)

print("CartPole-v1 funciona bien.")
print("Observación:", obs)
print("Reward:", reward)
print("Done:", terminated or truncated)

env.close()

Gymnasium: 1.0.0
Torch: 2.10.0+cu128
NumPy: 2.0.2
CUDA disponible: True
CartPole-v1 funciona bien.
Observación: [ 0.04224927  0.16234781  0.04864146 -0.25758597]
Reward: 1.0
Done: False


In [ ]:
%%writefile ppo_cartpole.py
import argparse
import os
import random
import time

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions.categorical import Categorical
from torch.utils.tensorboard import SummaryWriter


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--env-id", type=str, default="CartPole-v1")
    parser.add_argument("--total-timesteps", type=int, default=25000)
    parser.add_argument("--learning-rate", type=float, default=2.5e-4)
    parser.add_argument("--num-envs", type=int, default=4)
    parser.add_argument("--num-steps", type=int, default=128)
    parser.add_argument("--anneal-lr", type=bool, default=True)
    parser.add_argument("--gamma", type=float, default=0.99)
    parser.add_argument("--gae-lambda", type=float, default=0.95)
    parser.add_argument("--num-minibatches", type=int, default=4)
    parser.add_argument("--update-epochs", type=int, default=4)
    parser.add_argument("--norm-adv", type=bool, default=True)
    parser.add_argument("--clip-coef", type=float, default=0.2)
    parser.add_argument("--clip-vloss", type=bool, default=True)
    parser.add_argument("--ent-coef", type=float, default=0.01)
    parser.add_argument("--vf-coef", type=float, default=0.5)
    parser.add_argument("--max-grad-norm", type=float, default=0.5)
    parser.add_argument("--seed", type=int, default=1)
    args = parser.parse_args()

    args.batch_size = args.num_envs * args.num_steps
    args.minibatch_size = args.batch_size // args.num_minibatches
    args.num_iterations = args.total_timesteps // args.batch_size
    return args


def make_env(env_id, seed, idx):
    def thunk():
        env = gym.make(env_id)
        env = gym.wrappers.RecordEpisodeStatistics(env)
        env.action_space.seed(seed + idx)
        env.observation_space.seed(seed + idx)
        return env
    return thunk


def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer


class Agent(nn.Module):
    def __init__(self, envs):
        super().__init__()
        obs_dim = int(np.array(envs.single_observation_space.shape).prod())
        action_dim = envs.single_action_space.n

        self.critic = nn.Sequential(
            layer_init(nn.Linear(obs_dim, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 1), std=1.0),
        )

        self.actor = nn.Sequential(
            layer_init(nn.Linear(obs_dim, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, action_dim), std=0.01),
        )

    def get_value(self, x):
        return self.critic(x)

    def get_action_and_value(self, x, action=None):
        logits = self.actor(x)
        probs = Categorical(logits=logits)

        if action is None:
            action = probs.sample()

        return action, probs.log_prob(action), probs.entropy(), self.critic(x)


if __name__ == "__main__":
    args = parse_args()

    run_name = f"{args.env_id}__ppo__{args.seed}__{int(time.time())}"

    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Usando device:", device)

    writer = SummaryWriter(f"runs/{run_name}")
    writer.add_text(
        "hyperparameters",
        "|parametro|valor|\n|-|-|\n%s" % "\n".join([f"|{k}|{v}|" for k, v in vars(args).items()]),
    )

    envs = gym.vector.SyncVectorEnv(
        [make_env(args.env_id, args.seed, i) for i in range(args.num_envs)]
    )

    assert isinstance(envs.single_action_space, gym.spaces.Discrete), "Este PPO está hecho para acciones discretas."

    agent = Agent(envs).to(device)
    optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate, eps=1e-5)

    obs = torch.zeros((args.num_steps, args.num_envs) + envs.single_observation_space.shape).to(device)
    actions = torch.zeros((args.num_steps, args.num_envs) + envs.single_action_space.shape).to(device)
    logprobs = torch.zeros((args.num_steps, args.num_envs)).to(device)
    rewards = torch.zeros((args.num_steps, args.num_envs)).to(device)
    dones = torch.zeros((args.num_steps, args.num_envs)).to(device)
    values = torch.zeros((args.num_steps, args.num_envs)).to(device)

    global_step = 0
    start_time = time.time()

    next_obs, _ = envs.reset(seed=args.seed)
    next_obs = torch.Tensor(next_obs).to(device)
    next_done = torch.zeros(args.num_envs).to(device)

    episode_returns = np.zeros(args.num_envs)
    episode_lengths = np.zeros(args.num_envs)

    for iteration in range(1, args.num_iterations + 1):
        if args.anneal_lr:
            frac = 1.0 - (iteration - 1.0) / args.num_iterations
            lrnow = frac * args.learning_rate
            optimizer.param_groups[0]["lr"] = lrnow

        for step in range(args.num_steps):
            global_step += args.num_envs
            obs[step] = next_obs
            dones[step] = next_done

            with torch.no_grad():
                action, logprob, _, value = agent.get_action_and_value(next_obs)
                values[step] = value.flatten()

            actions[step] = action
            logprobs[step] = logprob

            next_obs_np, reward, terminated, truncated, info = envs.step(action.cpu().numpy())
            done = np.logical_or(terminated, truncated)

            episode_returns += reward
            episode_lengths += 1

            for i, d in enumerate(done):
                if d:
                    print(f"global_step={global_step}, episodic_return={episode_returns[i]:.2f}, episodic_length={episode_lengths[i]:.0f}")
                    writer.add_scalar("charts/episodic_return", episode_returns[i], global_step)
                    writer.add_scalar("charts/episodic_length", episode_lengths[i], global_step)
                    episode_returns[i] = 0
                    episode_lengths[i] = 0

            rewards[step] = torch.tensor(reward).to(device).view(-1)
            next_obs = torch.Tensor(next_obs_np).to(device)
            next_done = torch.Tensor(done).to(device)

        with torch.no_grad():
            next_value = agent.get_value(next_obs).reshape(1, -1)
            advantages = torch.zeros_like(rewards).to(device)
            lastgaelam = 0

            for t in reversed(range(args.num_steps)):
                if t == args.num_steps - 1:
                    nextnonterminal = 1.0 - next_done
                    nextvalues = next_value
                else:
                    nextnonterminal = 1.0 - dones[t + 1]
                    nextvalues = values[t + 1]

                delta = rewards[t] + args.gamma * nextvalues * nextnonterminal - values[t]
                advantages[t] = lastgaelam = delta + args.gamma * args.gae_lambda * nextnonterminal * lastgaelam

            returns = advantages + values

        b_obs = obs.reshape((-1,) + envs.single_observation_space.shape)
        b_logprobs = logprobs.reshape(-1)
        b_actions = actions.reshape((-1,) + envs.single_action_space.shape)
        b_advantages = advantages.reshape(-1)
        b_returns = returns.reshape(-1)
        b_values = values.reshape(-1)

        b_inds = np.arange(args.batch_size)

        for epoch in range(args.update_epochs):
            np.random.shuffle(b_inds)

            for start in range(0, args.batch_size, args.minibatch_size):
                end = start + args.minibatch_size
                mb_inds = b_inds[start:end]

                _, newlogprob, entropy, newvalue = agent.get_action_and_value(
                    b_obs[mb_inds],
                    b_actions.long()[mb_inds]
                )

                logratio = newlogprob - b_logprobs[mb_inds]
                ratio = logratio.exp()

                mb_advantages = b_advantages[mb_inds]
                if args.norm_adv:
                    mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

                pg_loss1 = -mb_advantages * ratio
                pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - args.clip_coef, 1 + args.clip_coef)
                pg_loss = torch.max(pg_loss1, pg_loss2).mean()

                newvalue = newvalue.view(-1)

                if args.clip_vloss:
                    v_loss_unclipped = (newvalue - b_returns[mb_inds]) ** 2
                    v_clipped = b_values[mb_inds] + torch.clamp(
                        newvalue - b_values[mb_inds],
                        -args.clip_coef,
                        args.clip_coef,
                    )
                    v_loss_clipped = (v_clipped - b_returns[mb_inds]) ** 2
                    v_loss = 0.5 * torch.max(v_loss_unclipped, v_loss_clipped).mean()
                else:
                    v_loss = 0.5 * ((newvalue - b_returns[mb_inds]) ** 2).mean()

                entropy_loss = entropy.mean()
                loss = pg_loss - args.ent_coef * entropy_loss + args.vf_coef * v_loss

                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(agent.parameters(), args.max_grad_norm)
                optimizer.step()

        writer.add_scalar("charts/learning_rate", optimizer.param_groups[0]["lr"], global_step)
        writer.add_scalar("losses/value_loss", v_loss.item(), global_step)
        writer.add_scalar("losses/policy_loss", pg_loss.item(), global_step)
        writer.add_scalar("losses/entropy", entropy_loss.item(), global_step)

        print(f"Iteración {iteration}/{args.num_iterations} terminada. global_step={global_step}")

    os.makedirs(f"runs/{run_name}", exist_ok=True)
    model_path = f"runs/{run_name}/agent.pt"
    torch.save(agent.state_dict(), model_path)

    print("Entrenamiento terminado.")
    print("Modelo guardado en:", model_path)

    envs.close()
    writer.close()

Writing ppo_cartpole.py


In [ ]:
!python ppo_cartpole.py --env-id CartPole-v1 --total-timesteps 2048

2026-05-20 03:55:41.110593: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Usando device: cuda
global_step=52, episodic_return=13.00, episodic_length=13
global_step=56, episodic_return=14.00, episodic_length=14
global_step=76, episodic_return=19.00, episodic_length=19
global_step=96, episodic_return=24.00, episodic_length=24
global_step=100, episodic_return=10.00, episodic_length=11
global_step=140, episodic_return=15.00, episodic_length=16
global_step=164, episodic_return=15.00, episodic_length=16
global_step=212, episodic_return=11.00, episodic_length=12
global_step=240, episodic_return=35.00, episodic_length=36
global_step=264, episodic_return=52.00, episodic_length=53
global_step=288, episodic_return=36.00, episodic_length=37
global_step=356

In [ ]:
!python ppo_cartpole.py --env-id CartPole-v1 --total-timesteps 25000

2026-05-20 04:02:10.314232: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Usando device: cuda
global_step=52, episodic_return=13.00, episodic_length=13
global_step=56, episodic_return=14.00, episodic_length=14
global_step=76, episodic_return=19.00, episodic_length=19
global_step=96, episodic_return=24.00, episodic_length=24
global_step=100, episodic_return=10.00, episodic_length=11
global_step=140, episodic_return=15.00, episodic_length=16
global_step=164, episodic_return=15.00, episodic_length=16
global_step=212, episodic_return=11.00, episodic_length=12
global_step=240, episodic_return=35.00, episodic_length=36
global_step=264, episodic_return=52.00, episodic_length=53
global_step=288, episodic_return=36.00, episodic_length=37
global_step=356

In [ ]:
import os
import glob
import torch
import gymnasium as gym
import numpy as np

from ppo_cartpole import Agent, make_env

modelos = glob.glob("runs/CartPole-v1__ppo__*/agent.pt")
modelos = sorted(modelos, key=os.path.getmtime)

model_path = modelos[-1]
print("Usando modelo:", model_path)

envs = gym.vector.SyncVectorEnv([make_env("CartPole-v1", 1, 0)])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
agent = Agent(envs).to(device)
agent.load_state_dict(torch.load(model_path, map_location=device))
agent.eval()

eval_env = gym.make("CartPole-v1")

num_episodes = 10
returns = []

for ep in range(num_episodes):
    obs, info = eval_env.reset()
    done = False
    total_reward = 0

    while not done:
        obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = agent.actor(obs_tensor)
            action = torch.argmax(logits, dim=1).item()

        obs, reward, terminated, truncated, info = eval_env.step(action)
        done = terminated or truncated
        total_reward += reward

    returns.append(total_reward)
    print(f"Episodio {ep + 1}: reward = {total_reward}")

eval_env.close()
envs.close()

print("\nReward promedio:", np.mean(returns))
print("Reward máximo:", np.max(returns))
print("Reward mínimo:", np.min(returns))

Usando modelo: runs/CartPole-v1__ppo__1__1779249734/agent.pt
Episodio 1: reward = 161.0
Episodio 2: reward = 276.0
Episodio 3: reward = 270.0
Episodio 4: reward = 235.0
Episodio 5: reward = 214.0
Episodio 6: reward = 191.0
Episodio 7: reward = 447.0
Episodio 8: reward = 224.0
Episodio 9: reward = 290.0
Episodio 10: reward = 299.0

Reward promedio: 260.7
Reward máximo: 447.0
Reward mínimo: 161.0


In [ ]:
!python ppo_cartpole.py --env-id CartPole-v1 --total-timesteps 100000

Usando device: cuda
global_step=52, episodic_return=13.00, episodic_length=13
global_step=56, episodic_return=14.00, episodic_length=14
global_step=76, episodic_return=19.00, episodic_length=19
global_step=96, episodic_return=24.00, episodic_length=24
global_step=100, episodic_return=10.00, episodic_length=11
global_step=140, episodic_return=15.00, episodic_length=16
global_step=164, episodic_return=15.00, episodic_length=16
global_step=212, episodic_return=11.00, episodic_length=12
global_step=240, episodic_return=35.00, episodic_length=36
global_step=264, episodic_return=52.00, episodic_length=53
global_step=288, episodic_return=36.00, episodic_length=37
global_step=356, episodic_return=35.00, episodic_length=36
global_step=360, episodic_return=29.00, episodic_length=30
global_step=388, episodic_return=30.00, episodic_length=31
global_step=396, episodic_return=26.00, episodic_length=27
global_step=400, episodic_return=10.00, episodic_length=11
global_step=400, episodic_return=9.00, e

In [ ]:
import glob
import os

modelos = glob.glob("runs/CartPole-v1__ppo__*/agent.pt")
modelos = sorted(modelos, key=os.path.getmtime)

for m in modelos:
    print(m, " | modificado:", os.path.getmtime(m))

runs/CartPole-v1__ppo__1__1779249345/agent.pt  | modificado: 1779249353.6812809
runs/CartPole-v1__ppo__1__1779249734/agent.pt  | modificado: 1779249752.6041017
runs/CartPole-v1__ppo__1__1779250231/agent.pt  | modificado: 1779250298.8160627


In [ ]:
import os
import glob
import torch
import gymnasium as gym
import numpy as np

from ppo_cartpole import Agent, make_env

# Buscar el modelo más reciente
modelos = glob.glob("runs/CartPole-v1__ppo__*/agent.pt")
modelos = sorted(modelos, key=os.path.getmtime)

print("Modelos encontrados:")
for m in modelos:
    print(m)

model_path = modelos[-1]
print("\nUsando modelo más reciente:", model_path)

# Crear agente
envs = gym.vector.SyncVectorEnv([make_env("CartPole-v1", 1, 0)])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
agent = Agent(envs).to(device)

agent.load_state_dict(torch.load(model_path, map_location=device))
agent.eval()

# Evaluación
eval_env = gym.make("CartPole-v1")

num_episodes = 10
returns = []

for ep in range(num_episodes):
    obs, info = eval_env.reset()
    done = False
    total_reward = 0

    while not done:
        obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = agent.actor(obs_tensor)
            action = torch.argmax(logits, dim=1).item()

        obs, reward, terminated, truncated, info = eval_env.step(action)
        done = terminated or truncated
        total_reward += reward

    returns.append(total_reward)
    print(f"Episodio {ep + 1}: reward = {total_reward}")

eval_env.close()
envs.close()

print("\nReward promedio:", np.mean(returns))
print("Reward máximo:", np.max(returns))
print("Reward mínimo:", np.min(returns))

Modelos encontrados:
runs/CartPole-v1__ppo__1__1779249345/agent.pt
runs/CartPole-v1__ppo__1__1779249734/agent.pt
runs/CartPole-v1__ppo__1__1779250231/agent.pt

Usando modelo más reciente: runs/CartPole-v1__ppo__1__1779250231/agent.pt
Episodio 1: reward = 500.0
Episodio 2: reward = 500.0
Episodio 3: reward = 358.0
Episodio 4: reward = 500.0
Episodio 5: reward = 500.0
Episodio 6: reward = 500.0
Episodio 7: reward = 423.0
Episodio 8: reward = 492.0
Episodio 9: reward = 206.0
Episodio 10: reward = 224.0

Reward promedio: 420.3
Reward máximo: 500.0
Reward mínimo: 206.0


In [ ]:
import os
import glob
import math
import torch
import gymnasium as gym
import numpy as np
import imageio

from PIL import Image, ImageDraw
from IPython.display import Video, display

from ppo_cartpole import Agent, make_env

# Buscar modelo más reciente
modelos = glob.glob("runs/CartPole-v1__ppo__*/agent.pt")
modelos = sorted(modelos, key=os.path.getmtime)

model_path = modelos[-1]
print("Usando modelo:", model_path)

# Crear agente
envs = gym.vector.SyncVectorEnv([make_env("CartPole-v1", 1, 0)])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
agent = Agent(envs).to(device)

agent.load_state_dict(torch.load(model_path, map_location=device))
agent.eval()

# Entorno SIN render_mode para evitar pygame
env = gym.make("CartPole-v1")

def dibujar_cartpole(obs, width=600, height=400):
    x, x_dot, theta, theta_dot = obs

    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)

    # Escala del mundo de CartPole
    world_width = 4.8
    scale = width / world_width

    cart_y = int(height * 0.65)
    cart_w = 60
    cart_h = 30

    cart_x = int(width / 2 + x * scale)

    # Suelo
    draw.line((0, cart_y + cart_h // 2, width, cart_y + cart_h // 2), fill="black", width=2)

    # Carrito
    left = cart_x - cart_w // 2
    right = cart_x + cart_w // 2
    top = cart_y - cart_h // 2
    bottom = cart_y + cart_h // 2
    draw.rectangle((left, top, right, bottom), outline="black", fill="gray")

    # Poste
    pole_len = 130
    pole_x0 = cart_x
    pole_y0 = top

    pole_x1 = pole_x0 + pole_len * math.sin(theta)
    pole_y1 = pole_y0 - pole_len * math.cos(theta)

    draw.line((pole_x0, pole_y0, pole_x1, pole_y1), fill="blue", width=8)
    draw.ellipse((pole_x0 - 5, pole_y0 - 5, pole_x0 + 5, pole_y0 + 5), fill="red")

    return np.array(img)

frames = []
obs, info = env.reset()
done = False
total_reward = 0

while not done:
    frames.append(dibujar_cartpole(obs))

    obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = agent.actor(obs_tensor)
        action = torch.argmax(logits, dim=1).item()

    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    total_reward += reward

env.close()
envs.close()

video_path = "cartpole_ppo_agent_sin_pygame.mp4"
imageio.mimsave(video_path, frames, fps=30)

print("Video guardado en:", video_path)
print("Reward del episodio grabado:", total_reward)

display(Video(video_path, embed=True))

Usando modelo: runs/CartPole-v1__ppo__1__1779250231/agent.pt


Video guardado en: cartpole_ppo_agent_sin_pygame.mp4
Reward del episodio grabado: 500.0


In [ ]:
import os
import glob
import shutil

# Buscar modelo más reciente
modelos = glob.glob("runs/CartPole-v1__ppo__*/agent.pt")
modelos = sorted(modelos, key=os.path.getmtime)
modelo_final = modelos[-1]

video_final = "cartpole_ppo_agent_sin_pygame.mp4"

print("Modelo final:", modelo_final)
print("Existe modelo:", os.path.exists(modelo_final))
print("Video final:", video_final)
print("Existe video:", os.path.exists(video_final))

# Crear carpeta de entrega
os.makedirs("entrega_ppo_cartpole", exist_ok=True)

shutil.copy(modelo_final, "entrega_ppo_cartpole/agent.pt")
shutil.copy(video_final, "entrega_ppo_cartpole/cartpole_ppo_agent_sin_pygame.mp4")
shutil.copy("ppo_cartpole.py", "entrega_ppo_cartpole/ppo_cartpole.py")

shutil.make_archive("entrega_ppo_cartpole", "zip", "entrega_ppo_cartpole")

print("ZIP creado: entrega_ppo_cartpole.zip")

Modelo final: runs/CartPole-v1__ppo__1__1779250231/agent.pt
Existe modelo: True
Video final: cartpole_ppo_agent_sin_pygame.mp4
Existe video: True
ZIP creado: entrega_ppo_cartpole.zip


In [ ]:
from google.colab import files

files.download("entrega_ppo_cartpole.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

To be able to share your model with the community there are three more steps to follow:

1️⃣ (If it's not already done) create an account to HF ➡ https://huggingface.co/join

2️⃣ Sign in and then, you need to store your authentication token from the Hugging Face website.
- Create a new token (https://huggingface.co/settings/tokens) **with write role**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

- Copy the token
- Run the cell below and paste the token

In [ ]:
from huggingface_hub import notebook_login
notebook_login()
!git config --global credential.helper store

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import os
import json

repo_id = "Lizeth-otalora07/ppo-CartPole-v1"

readme = """
---
library_name: pytorch
tags:
- reinforcement-learning
- ppo
- cartpole
- gymnasium
- deep-rl-course
---

# PPO Agent - CartPole-v1

This repository contains a PPO agent trained on `CartPole-v1` using Gymnasium and PyTorch.

## Environment

- Environment: `CartPole-v1`
- Algorithm: PPO
- Framework: PyTorch
- Gym library: Gymnasium

## Results

The trained agent achieved:

- Average reward: 420.3
- Maximum reward: 500.0
- Minimum reward: 206.0

A recorded episode reached the maximum reward of 500.0.

## Files

- `agent.pt`: trained model weights
- `ppo_cartpole.py`: training script
- `cartpole_ppo_agent_sin_pygame.mp4`: video of the trained agent
"""

with open("entrega_ppo_cartpole/README.md", "w", encoding="utf-8") as f:
    f.write(readme)

requirements = """
torch
gymnasium==1.0.0
numpy
tensorboard
huggingface_hub
wasabi
imageio
imageio-ffmpeg
Pillow
"""

with open("entrega_ppo_cartpole/requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements)

config = {
    "env_id": "CartPole-v1",
    "algorithm": "PPO",
    "framework": "PyTorch",
    "reward_average": 420.3,
    "reward_max": 500.0,
    "reward_min": 206.0
}

with open("entrega_ppo_cartpole/config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print("Carpeta lista para subir:", repo_id)

Carpeta lista para subir: Lizeth-otalora07/ppo-CartPole-v1


In [ ]:
from huggingface_hub import HfApi, upload_folder

repo_id = "Lizeth-otalora07/ppo-CartPole-v1"

api = HfApi()

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True
)

upload_folder(
    folder_path="entrega_ppo_cartpole",
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload PPO CartPole trained agent"
)

print("Subido correctamente a Hugging Face:")
print(f"https://huggingface.co/{repo_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ega_ppo_cartpole/agent.pt: 100%|##########| 40.9kB / 40.9kB            

Subido correctamente a Hugging Face:
https://huggingface.co/Lizeth-otalora07/ppo-CartPole-v1


If you don't want to use a Google Colab or a Jupyter Notebook, you need to use this command instead: `huggingface-cli login`

## Let's start the training 🔥
- ⚠️ ⚠️ ⚠️  Don't use **the same repo id with the one you used for the Unit 1**
- Now that you've coded from scratch PPO and added the Hugging Face Integration, we're ready to start the training 🔥

- First, you need to copy all your code to a file you create called `ppo.py`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit9/step1.png" alt="PPO"/>

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit9/step2.png" alt="PPO"/>

- Now we just need to run this python script using `python <name-of-python-script>.py` with the additional parameters we defined with `argparse`

- You should modify more hyperparameters otherwise the training will not be super stable.

In [ ]:
!python ppo.py --env-id="LunarLander-v2" --repo-id="YOUR_REPO_ID" --total-timesteps=50000

python3: can't open file '/content/ppo.py': [Errno 2] No such file or directory


## Some additional challenges 🏆
The best way to learn **is to try things by your own**! Why not trying  another environment?


See you on Unit 8, part 2 where we going to train agents to play Doom 🔥
## Keep learning, stay awesome 🤗